<a href="https://colab.research.google.com/github/kds825/education/blob/main/GPU_%EC%9A%A9%EB%9F%89_%EA%B3%84%EC%82%B0_%ED%95%99%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi


Sun Sep 27 04:52:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

print(
    "GPU 사용 가능:",
    torch.cuda.is_available()
)

gpu_name = torch.cuda.get_device_name(0)
print(
    "GPU:", gpu_name
)

total_vram =(
    torch.cuda.get_device_properties(0)
    .total_memory
)

total_vram_gb = (
    total_vram / 1024**3
)

print(
    "VRAM:",
    round(total_vram_gb, 2),
    "GB"
)


GPU 사용 가능: True
GPU: Tesla T4
VRAM: 14.56 GB


In [3]:
!pip install -q transformers accelerate

In [4]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

import torch
import time

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda"
)



config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [5]:
used_vram = (
    torch.cuda.memory_allocated()
    /1024**3
)

print(
    "현재 사용 VRAM:",
    round(used_vram, 2),
    "GB"
)

현재 사용 VRAM: 0.92 GB


In [6]:
prompt ="""
당신은 회사의 재무분석 AI Agent입니다.

다음 정보를 분석하세요.

매출액: 1,000억원
전년 매출액: 800억원
영업이익: 120억원

매출 성장률과 영업이익률을 계산하고
간단한 분석을 작성하세요.
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)
inputs = inputs.to("cuda")


In [7]:
start_time = time.time()
with torch.no_grad():
  outputs =model.generate(
      **inputs,
      max_new_tokens=1000
  )

torch.cuda.synchronize()
end_time = time.time()

elapsed_time = (
    end_time - start_time
)

print(
    "응답 시간:",
    round(elapsed_time, 2),
    "초"
)

응답 시간: 82.43 초


In [8]:
input_tokens = inputs["input_ids"].shape[1]
total_tokens = outputs.shape[1]
generated_tokens=(
    total_tokens - input_tokens
)

print(
    "입력 Token:",
    input_tokens
)

print(
    "생성 Token:",
    generated_tokens
)

tokens_per_second = (
    generated_tokens/elapsed_time
)

print(
    "생성 속도:",
    round(tokens_per_second,2),
    "tokens/sec"
)

입력 Token: 86
생성 Token: 625
생성 속도: 7.58 tokens/sec


In [9]:
def call_llm(prompt):
  inputs =tokenizer(
      prompt,
      return_tensors = "pt"
  ).to("cuda")
  with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

  result = tokenizer.decode(
      outputs[0],
      skip_special_tokens = True
  )
  return result


In [10]:
start =time.time()
result1=call_llm(
    "사용자의 재무 질문 의도를 분석하라"
)

result2=call_llm(
    "매출 성장률과 영업이익률을 분석하는 방법을 설명해라."
)

result3 = call_llm(
    "재무 분석 결과를 경영진에게 간단하게 설명해라"
)

torch.cuda.synchronize()
end = time.time()

print(
    "Agent 전체 처리시간:",
    round(end-start,2),
    "초"
)

Agent 전체 처리시간: 14.27 초


In [12]:
import time
number_of_users = 5
start = time.time()
for user in range(number_of_users):
  with torch.no_grad():
    output=model.generate(
        **inputs,
        max_new_tokens = 100
    )

torch.cuda.synchronize()
end = time.time()
total_time = end - start

print(
    "사용자 수:",
    number_of_users
)

print(
    "전체 처리 시간:",
    round(total_time, 2),
    "초"
)

print(
    "사용자당 평균:",
    round(total_time / number_of_users, 2),
    "초"
)



사용자 수: 5
전체 처리 시간: 18.97 초
사용자당 평균: 3.79 초


In [13]:
prompts = [

    "매출이 감소한 주요 원인을 분석해줘.",

    "영업이익률이 감소한 이유를 분석해줘.",

    "재고자산이 증가하면 어떤 위험이 있어?",

    "매출채권이 증가하면 어떤 위험이 있어?",

    "현금흐름이 악화되는 원인을 설명해줘."
]

batch_inputs = tokenizer(
    prompts,
    return_tensors = "pt",
    padding=True
)

batch_inputs = batch_inputs.to(model.device)

In [15]:
import time
import torch

torch.cuda.synchronize()
start = time.time()
with torch.no_grad():
  batch_outputs = model.generate(
      **batch_inputs,
      max_new_tokens = 1000
  )

torch.cuda.synchronize()
end = time.time()
total_time = end-start

print(
    "사용자 수:",
    len(prompts)
)

print(
    "Batch 전체 처리시간:",
    round(total_time, 2),
    "초"
)

number_of_users = len(prompts)
requests_per_second = (
    number_of_users/ total_time
)

print(
    "처리량:",
    round(requests_per_second, 2),
    "requests/sec"
)

[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


사용자 수: 5
Batch 전체 처리시간: 70.72 초
처리량: 0.07 requests/sec


In [17]:
import time
import torch

def test_users(number_of_users):
  prompts =[
      "기업의 주요 재무위험을 간단히 분석해줘."
  ] * number_of_users

  batch_inputs = tokenizer(
      prompts,
      return_tensors="pt",
      padding = True
  )

  batch_inputs = batch_inputs.to(
      model.device
  )
  torch.cuda.reset_peak_memory_stats()
  torch.cuda.synchronize()

  start = time.time()

  with torch.no_grad():
    outputs = model.generate(
        **batch_inputs,
        max_new_tokens=1000
    )
  torch.cuda.synchronize()
  end = time.time()
  elapsed = end-start

  peak_vram = (
      torch.cuda.max_memory_allocated()/1024**3
  )

  throughput = (
      number_of_users/elapsed
  )
  print(
      "사용자:", number_of_users,
      "|시간:", round(elapsed, 2),"초",
      "|처리량:", round(throughput, 2),"req/s"
      "|Peak VRAM:", round(peak_vram, 2), "GB"
  )

In [20]:
test_users(1)
test_users(2)
test_users(5)
test_users(10)

사용자: 1 |시간: 77.4 초 |처리량: 0.01 req/s|Peak VRAM: 0.97 GB
사용자: 2 |시간: 83.26 초 |처리량: 0.02 req/s|Peak VRAM: 0.99 GB
사용자: 5 |시간: 82.07 초 |처리량: 0.06 req/s|Peak VRAM: 1.07 GB
사용자: 10 |시간: 84.65 초 |처리량: 0.12 req/s|Peak VRAM: 1.19 GB
